<a href="https://colab.research.google.com/github/Tim-Albiges/Tim-Albiges/blob/main/Gradio_Exploratory_and_Correlation_Analysis_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import nest_asyncio
import gradio as gr
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, spearmanr
import tempfile
import warnings

In [5]:
# Ignore warnings for cleaner UI output
warnings.filterwarnings('ignore')
# Allows Gradio to share Colab's event loop safely
nest_asyncio.apply()
# Ensure gradio is all closed before opening a new connection
gr.close_all()

In [6]:
# ---------------------------------
# CLASS: Data Science Logic Core
# ---------------------------------
class DataAnalyzer:
    """
    State manager for the Data Analysis Application.
    Handles data loading, type inference, statistical calculations, and audit trails.
    """
    def __init__(self):
        self.df = None
        self.column_types = {}  # Stores 'Numeric' or 'Categorical'

    def load_data(self, file_obj):
        """Loads data from CSV or Excel."""
        if file_obj is None:
            return None, "No file uploaded."

        try:
            file_path = file_obj.name
            if file_path.endswith('.csv'):
                self.df = pd.read_csv(file_path)
            elif file_path.endswith(('.xls', '.xlsx')):
                self.df = pd.read_excel(file_path)
            else:
                return None, "Unsupported file format. Please use CSV or Excel."

            # Initial Type Inference
            self._infer_types()
            return self.df.head(), f"Loaded successfully: {self.df.shape[0]} rows, {self.df.shape[1]} columns."
        except Exception as e:
            return None, f"Error loading file: {str(e)}"

    def _infer_types(self):
        """
        Heuristic to automatically detect data types.
        """
        self.column_types = {}
        for col in self.df.columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                # If numeric but < 10 unique values, treat as Categorical (e.g. Rank 1-5)
                if self.df[col].nunique() < 10:
                    self.column_types[col] = 'Categorical'
                else:
                    self.column_types[col] = 'Numeric'
            else:
                self.column_types[col] = 'Categorical'

    def update_type(self, col_name, new_type):
        """Manually override the data type."""
        self.column_types[col_name] = new_type

    def get_column_names(self):
        return list(self.df.columns) if self.df is not None else []

    def get_eda_plots(self, selected_columns):
        """Generates Distribution plots and Frequency tables with safeguards."""
        if self.df is None or not selected_columns:
            return [], None, "No data or columns selected."

        plots = []
        metrics_list = []

        for col in selected_columns:
            plt.figure(figsize=(8, 4))
            dtype = self.column_types.get(col, 'Numeric')

            # --- Analysis for Numeric (Continuous/Ratio) ---
            if dtype == 'Numeric':
                # SAFEGUARD: Coerce to numeric. Any text (like "M" or "PT_001") becomes NaN
                safe_col = pd.to_numeric(self.df[col], errors='coerce')

                # Check if coercion resulted in an entirely empty/NaN column
                if safe_col.dropna().empty:
                    # Fallback gracefully if the user forced a text column to be numeric
                    sns.histplot(self.df[col].astype(str), color='salmon')
                    plt.title(f"Distribution: {col} (Invalid Numeric Data)")
                    metrics_list.append({
                        'Feature': col, 'Type': 'Numeric (Failed)',
                        'Mean': 'N/A', 'Median': 'N/A', 'Std': 'N/A', 'Min': 'N/A', 'Max': 'N/A'
                    })
                else:
                    sns.histplot(safe_col, kde=True, color='skyblue')
                    plt.title(f"Distribution: {col} (Numeric)")
                    desc = safe_col.describe()

                    # SAFEGUARD: Use .get() to prevent KeyError if stats are missing
                    metrics_list.append({
                        'Feature': col, 'Type': 'Numeric',
                        'Mean': round(desc.get('mean', np.nan), 2),
                        'Median': round(desc.get('50%', np.nan), 2),
                        'Std': round(desc.get('std', np.nan), 2),
                        'Min': desc.get('min', 'N/A'),
                        'Max': desc.get('max', 'N/A')
                    })

            # --- Analysis for Categorical/Binary/Ordinal ---
            else:
                top_counts = self.df[col].value_counts().head(20)
                sns.barplot(x=top_counts.index, y=top_counts.values, palette='viridis')
                plt.title(f"Frequency: {col} (Categorical)")
                plt.xticks(rotation=45)

                metrics_list.append({
                    'Feature': col, 'Type': 'Categorical',
                    'Unique Values': self.df[col].nunique(),
                    'Top Value': top_counts.index[0] if not top_counts.empty else 'N/A',
                    'Top Freq': top_counts.iloc[0] if not top_counts.empty else 'N/A'
                })

            plt.tight_layout()
            tmp_path = tempfile.mktemp(suffix='.png')
            plt.savefig(tmp_path)
            plt.close()
            plots.append(tmp_path)

        metrics_df = pd.DataFrame(metrics_list)
        metrics_csv = tempfile.mktemp(suffix='.csv')
        metrics_df.to_csv(metrics_csv, index=False)

        return plots, metrics_csv, "EDA generated successfully."

    def calculate_smart_correlation(self, selected_columns):
        """
        Performs correlation analysis based on data types and LOGS the methodology.
        """
        if self.df is None or len(selected_columns) < 2:
            return None, None, None, "Select at least 2 columns."

        sub_df = self.df[selected_columns].dropna()
        corr_matrix = pd.DataFrame(index=selected_columns, columns=selected_columns, dtype=float)

        # --- NEW: Methodology Logging List ---
        methodology_log = []

        for col1 in selected_columns:
            for col2 in selected_columns:
                if col1 == col2:
                    corr_matrix.loc[col1, col2] = 1.0
                    continue

                # Avoid duplicate logging (e.g., A vs B and B vs A)
                # We process the calculation for the matrix, but we only log unique pairs

                type1 = self.column_types.get(col1)
                type2 = self.column_types.get(col2)

                test_name = "Unknown"
                val = 0.0

                # 1. Num vs Num (Spearman)
                if type1 == 'Numeric' and type2 == 'Numeric':
                    test_name = "Spearman Rank (Non-Parametric)"
                    rho, _ = spearmanr(sub_df[col1], sub_df[col2])
                    val = rho

                # 2. Cat vs Cat (Cramer's V)
                elif type1 == 'Categorical' and type2 == 'Categorical':
                    test_name = "Cramer's V (Chi-Square based)"
                    confusion_matrix = pd.crosstab(sub_df[col1], sub_df[col2])
                    chi2 = chi2_contingency(confusion_matrix)[0]
                    n = confusion_matrix.sum().sum()
                    phi2 = chi2 / n
                    r, k = confusion_matrix.shape
                    with np.errstate(divide='ignore', invalid='ignore'):
                        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
                        rcorr = r - ((r-1)**2)/(n-1)
                        kcorr = k - ((k-1)**2)/(n-1)
                        if min((kcorr-1), (rcorr-1)) == 0:
                            val = 0
                        else:
                            val = np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

                # 3. Mixed (Correlation Ratio / Eta)
                else:
                    test_name = "Correlation Ratio (Eta)"
                    cat_col = col1 if type1 == 'Categorical' else col2
                    num_col = col2 if type1 == 'Categorical' else col1

                    groups = sub_df.groupby(cat_col)[num_col]
                    ss_between = sum(groups.count() * (groups.mean() - sub_df[num_col].mean())**2)
                    ss_total = sum((sub_df[num_col] - sub_df[num_col].mean())**2)
                    val = np.sqrt(ss_between / ss_total) if ss_total != 0 else 0

                corr_matrix.loc[col1, col2] = val

                # Log the methodology (Only log once per pair to keep the report clean)
                # We check if the reverse pair is already in the log? No, easier to just dump all for verification.
                methodology_log.append({
                    'Feature 1': col1,
                    'Type 1': type1,
                    'Feature 2': col2,
                    'Type 2': type2,
                    'Method Applied': test_name,
                    'Correlation Coefficient': round(val, 4)
                })

        # --- Generate Heatmap ---
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix.astype(float), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
        plt.title("Mixed-Type Correlation Matrix")
        plt.tight_layout()

        heatmap_path = tempfile.mktemp(suffix='.png')
        plt.savefig(heatmap_path)
        plt.close()

        # --- Save Correlation Matrix ---
        csv_path = tempfile.mktemp(suffix='.csv')
        corr_matrix.to_csv(csv_path)

        # --- Save Methodology Report ---
        method_df = pd.DataFrame(methodology_log)
        method_path = tempfile.mktemp(suffix='.csv')
        method_df.to_csv(method_path, index=False)

        return heatmap_path, csv_path, method_path, "Correlation Analysis & Methodology Report Complete."


In [7]:
# -------------------------------------------------------------------------
# UI: Gradio Interface Construction
# -------------------------------------------------------------------------

analyzer = DataAnalyzer()

def on_file_upload(file):
    preview, msg = analyzer.load_data(file)
    cols = analyzer.get_column_names()
    type_df = pd.DataFrame(list(analyzer.column_types.items()), columns=['Feature', 'Detected Type'])
    return preview, msg, gr.update(choices=cols), gr.update(choices=cols), gr.update(choices=cols), type_df

def change_column_type(col_name, new_type):
    if not col_name: return "Select a column first."
    analyzer.update_type(col_name, new_type)
    type_df = pd.DataFrame(list(analyzer.column_types.items()), columns=['Feature', 'Current Type'])
    return type_df

def run_eda(selected_cols):
    return analyzer.get_eda_plots(selected_cols)

def run_correlation(selected_cols):
    return analyzer.calculate_smart_correlation(selected_cols)

# Define the Gradio Blocks
with gr.Blocks(title="Auto-EDA & Correlation Tool", theme=gr.themes.Soft()) as app:
    gr.Markdown("## 📊 Intelligent Data Science & EDA Tool")
    gr.Markdown("Upload your dataset to automatically detect types, visualize distributions, and perform mixed-type correlation analysis.")

    with gr.Tab("1. Load & Refine Data"):
        with gr.Row():
            file_input = gr.File(label="Upload CSV or Excel", file_types=[".csv", ".xlsx"])
            status_output = gr.Textbox(label="Status")

        with gr.Row():
            data_preview = gr.Dataframe(label="Data Preview (First 5 Rows)")

        gr.Markdown("### 🛠️ Data Type Verification")
        with gr.Row():
            type_table = gr.Dataframe(label="Feature Data Types")
            with gr.Column():
                col_selector = gr.Dropdown(label="Select Feature to Edit")
                new_type_selector = gr.Radio(["Numeric", "Categorical"], label="Set Correct Type")
                update_btn = gr.Button("Update Type")

        update_btn.click(change_column_type, inputs=[col_selector, new_type_selector], outputs=[type_table])

    with gr.Tab("2. Exploratory Analysis (EDA)"):
        gr.Markdown("Select features to visualize distributions and see frequency counts.")
        eda_col_selector = gr.Dropdown(label="Select Features for EDA", multiselect=True)
        eda_btn = gr.Button("Generate EDA Reports")

        with gr.Row():
            plot_gallery = gr.Gallery(label="Distributions & Counts")
            with gr.Column():
                stats_file = gr.File(label="Download Descriptive Stats (CSV)")
                eda_status = gr.Textbox(label="Log", interactive=False)

        eda_btn.click(run_eda, inputs=[eda_col_selector], outputs=[plot_gallery, stats_file, eda_status])

    with gr.Tab("3. Correlation Analysis"):
        gr.Markdown("### 🔗 Smart Correlation Matrix")
        gr.Markdown("This tool selects statistical tests based on data types. Download the **Methodology Report** below for full details.")

        corr_col_selector = gr.Dropdown(label="Select Features for Correlation", multiselect=True)
        corr_btn = gr.Button("Calculate Correlation")

        with gr.Row():
            heatmap_img = gr.Image(label="Correlation Heatmap")
            with gr.Column():
                # Added the new export button/file slot here
                corr_file = gr.File(label="Download Matrix (CSV)")
                method_file = gr.File(label="Download Methodology Report (CSV)")
                corr_status = gr.Textbox(label="Log", interactive=False)

        corr_btn.click(run_correlation, inputs=[corr_col_selector], outputs=[heatmap_img, corr_file, method_file, corr_status])

    file_input.upload(
        on_file_upload,
        inputs=[file_input],
        outputs=[data_preview, status_output, col_selector, eda_col_selector, corr_col_selector, type_table]
    )



In [8]:
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a3c50e147c0710adf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
